# Module 8: Object-Oriented Programming for Machine Learning

**Topics Covered:**
- Classes, `__init__`, attributes, methods
- Inheritance and `super()`
- Dunder methods (`__repr__`, `__len__`, `__getitem__`)
- Properties and encapsulation
- Custom sklearn-compatible Transformers (`BaseEstimator`, `TransformerMixin`)
- Custom sklearn-compatible Estimators
- Implementing a Dataset class (PyTorch-style)
- Design patterns in ML frameworks

> OOP is how ML libraries are built. Understanding it lets you extend any framework, write custom components, and read source code.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin, ClassifierMixin
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score
from sklearn.datasets import make_classification
from sklearn.utils.validation import check_X_y, check_array, check_is_fitted
import matplotlib.pyplot as plt

np.random.seed(42)
print("Libraries loaded.")

---
## 1. Classes — The Building Block

In [ ]:
# A simple ML model config class
class ModelConfig:
    """Stores and validates hyperparameters for an ML model."""
    
    VALID_OPTIMIZERS = {'adam', 'sgd', 'rmsprop'}
    
    def __init__(self, name, learning_rate=0.001, n_epochs=100,
                 batch_size=32, optimizer='adam'):
        self.name = name
        self.learning_rate = learning_rate
        self.n_epochs = n_epochs
        self.batch_size = batch_size
        self.optimizer = optimizer
        self._validate()
    
    def _validate(self):
        if self.learning_rate <= 0:
            raise ValueError(f"learning_rate must be > 0, got {self.learning_rate}")
        if self.optimizer not in self.VALID_OPTIMIZERS:
            raise ValueError(f"optimizer must be one of {self.VALID_OPTIMIZERS}")
    
    def to_dict(self):
        return {
            'name': self.name,
            'lr': self.learning_rate,
            'epochs': self.n_epochs,
            'batch_size': self.batch_size,
            'optimizer': self.optimizer
        }
    
    def __repr__(self):
        return (f"ModelConfig({self.name!r}, lr={self.learning_rate}, "
                f"epochs={self.n_epochs}, optimizer={self.optimizer!r})")
    
    def __eq__(self, other):
        return self.to_dict() == other.to_dict()

cfg1 = ModelConfig('ResNet50', learning_rate=0.01, n_epochs=200)
cfg2 = ModelConfig('BERT', learning_rate=2e-5, n_epochs=3, batch_size=16, optimizer='adam')

print(cfg1)
print(cfg2)
print(f"Equal? {cfg1 == cfg2}")
print(f"Config dict: {cfg2.to_dict()}")

In [ ]:
# Properties — controlled attribute access
class TrainingMetrics:
    """Tracks training metrics and computes derived statistics."""
    
    def __init__(self):
        self._losses = []
        self._val_losses = []
    
    def log(self, loss, val_loss):
        self._losses.append(loss)
        self._val_losses.append(val_loss)
    
    @property
    def best_epoch(self):
        return int(np.argmin(self._val_losses)) + 1
    
    @property
    def best_val_loss(self):
        return min(self._val_losses) if self._val_losses else None
    
    @property
    def is_overfitting(self):
        if len(self._losses) < 5:
            return False
        recent_gap = np.mean(np.array(self._val_losses[-5:]) - np.array(self._losses[-5:]))
        return recent_gap > 0.1
    
    def __len__(self):
        return len(self._losses)
    
    def __repr__(self):
        return f"TrainingMetrics(epochs={len(self)}, best_val_loss={self.best_val_loss:.4f})"

metrics = TrainingMetrics()
for epoch in range(20):
    train_loss = 1.5 * np.exp(-0.15 * epoch) + np.random.randn() * 0.02
    val_loss   = 1.5 * np.exp(-0.10 * epoch) + 0.15 + np.random.randn() * 0.03
    metrics.log(train_loss, val_loss)

print(metrics)
print(f"Best epoch: {metrics.best_epoch}")
print(f"Best val loss: {metrics.best_val_loss:.4f}")
print(f"Overfitting? {metrics.is_overfitting}")

### Exercise 8.1 — Classes & Properties
Build an `ExperimentTracker` class that:
- `__init__`: takes `experiment_name` and `model_name`
- `log_run(params, metrics)`: stores a dict {params, metrics} for each run
- `best_run` property: returns the run with the highest `accuracy` in metrics
- `summary()` method: prints a table of all runs sorted by accuracy descending
- `__len__`: number of logged runs
- `__repr__`: meaningful string

In [ ]:
# YOUR CODE HERE


In [ ]:
# SOLUTION
class ExperimentTracker:
    """Tracks hyperparameter experiments and their results."""
    
    def __init__(self, experiment_name, model_name):
        self.experiment_name = experiment_name
        self.model_name = model_name
        self._runs = []
    
    def log_run(self, params, metrics):
        self._runs.append({'run_id': len(self._runs) + 1, 'params': params, 'metrics': metrics})
    
    @property
    def best_run(self):
        if not self._runs:
            return None
        return max(self._runs, key=lambda r: r['metrics'].get('accuracy', 0))
    
    def summary(self):
        print(f"\n{'='*60}")
        print(f"Experiment: {self.experiment_name} | Model: {self.model_name}")
        print(f"{'Run':>5} {'Params':<30} {'Accuracy':>10} {'F1':>8}")
        print('-' * 60)
        sorted_runs = sorted(self._runs, key=lambda r: r['metrics'].get('accuracy', 0), reverse=True)
        for run in sorted_runs:
            params_str = str(run['params'])[:28]
            acc = run['metrics'].get('accuracy', 0)
            f1  = run['metrics'].get('f1', 0)
            print(f"{run['run_id']:>5} {params_str:<30} {acc:>10.4f} {f1:>8.4f}")
    
    def __len__(self):
        return len(self._runs)
    
    def __repr__(self):
        return f"ExperimentTracker({self.experiment_name!r}, runs={len(self)})"

tracker = ExperimentTracker('Loan Default Study', 'RandomForest')
np.random.seed(42)
for n_est in [50, 100, 200]:
    for depth in [3, 5, None]:
        acc = np.random.uniform(0.80, 0.93)
        tracker.log_run(
            params={'n_estimators': n_est, 'max_depth': depth},
            metrics={'accuracy': acc, 'f1': acc - np.random.uniform(0, 0.03)}
        )

print(tracker)
print(f"Total runs: {len(tracker)}")
print(f"Best run: {tracker.best_run}")
tracker.summary()

---
## 2. Inheritance

In [ ]:
# Base class — shared behaviour
class BaseMLModel:
    """Base class for ML models — defines the interface."""
    
    def __init__(self, name, random_state=42):
        self.name = name
        self.random_state = random_state
        self.is_fitted = False
    
    def fit(self, X, y):
        raise NotImplementedError("Subclasses must implement fit()")
    
    def predict(self, X):
        if not self.is_fitted:
            raise RuntimeError("Model must be fitted before predicting.")
        raise NotImplementedError("Subclasses must implement predict()")
    
    def score(self, X, y):
        y_pred = self.predict(X)
        return np.mean(y_pred == y)
    
    def __repr__(self):
        status = "fitted" if self.is_fitted else "not fitted"
        return f"{self.__class__.__name__}({self.name!r}, {status})"


class KNNClassifier(BaseMLModel):
    """K-Nearest Neighbors classifier from scratch."""
    
    def __init__(self, k=3, random_state=42):
        super().__init__(name='KNN', random_state=random_state)
        self.k = k
    
    def fit(self, X, y):
        self.X_train = np.array(X)
        self.y_train = np.array(y)
        self.is_fitted = True
        return self
    
    def predict(self, X):
        super().predict(X)  # checks is_fitted
        X = np.array(X)
        predictions = []
        for x in X:
            dists = np.linalg.norm(self.X_train - x, axis=1)
            k_indices = np.argsort(dists)[:self.k]
            k_labels = self.y_train[k_indices]
            predictions.append(np.bincount(k_labels).argmax())
        return np.array(predictions)


class WeightedKNNClassifier(KNNClassifier):
    """KNN with inverse-distance weighting."""
    
    def predict(self, X):
        if not self.is_fitted:
            raise RuntimeError("Model must be fitted before predicting.")
        X = np.array(X)
        predictions = []
        for x in X:
            dists = np.linalg.norm(self.X_train - x, axis=1)
            k_indices = np.argsort(dists)[:self.k]
            k_dists   = dists[k_indices] + 1e-9
            weights   = 1.0 / k_dists
            k_labels  = self.y_train[k_indices]
            n_classes = len(np.unique(self.y_train))
            class_weights = np.zeros(n_classes)
            for label, w in zip(k_labels, weights):
                class_weights[label] += w
            predictions.append(np.argmax(class_weights))
        return np.array(predictions)

# Test
np.random.seed(42)
X_data, y_data = make_classification(n_samples=200, n_features=5, random_state=42)
X_tr, X_te = X_data[:160], X_data[160:]
y_tr, y_te = y_data[:160], y_data[160:]

knn = KNNClassifier(k=5)
wknn = WeightedKNNClassifier(k=5)

for model in [knn, wknn]:
    model.fit(X_tr, y_tr)
    print(f"{model} — Test accuracy: {model.score(X_te, y_te):.4f}")

---
## 3. Custom Sklearn-Compatible Transformers

In [ ]:
# Custom transformer that works in sklearn Pipelines
class OutlierClipper(BaseEstimator, TransformerMixin):
    """Clips outliers beyond n standard deviations from the mean.
    
    Works in sklearn Pipelines — fits statistics on train, applies to test.
    """
    
    def __init__(self, n_std=3.0):
        self.n_std = n_std
    
    def fit(self, X, y=None):
        X = check_array(X)
        self.mean_ = np.mean(X, axis=0)
        self.std_  = np.std(X, axis=0)
        self.lower_ = self.mean_ - self.n_std * self.std_
        self.upper_ = self.mean_ + self.n_std * self.std_
        return self
    
    def transform(self, X, y=None):
        check_is_fitted(self)
        X = check_array(X).copy()
        return np.clip(X, self.lower_, self.upper_)
    
    def __repr__(self):
        return f"OutlierClipper(n_std={self.n_std})"


class LogTransformer(BaseEstimator, TransformerMixin):
    """Applies log1p transform to specified columns (handles right-skewed features)."""
    
    def fit(self, X, y=None):
        check_array(X)
        return self
    
    def transform(self, X, y=None):
        X = check_array(X).copy()
        if np.any(X < 0):
            raise ValueError("LogTransformer requires non-negative values")
        return np.log1p(X)


# Use in a Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

pipe_custom = Pipeline([
    ('clip',   OutlierClipper(n_std=2.5)),
    ('scale',  StandardScaler()),
    ('clf',    LogisticRegression(max_iter=1000, random_state=42))
])

# Inject outliers to test
X_noisy = X_data.copy()
X_noisy[:5] *= 10  # extreme outliers

pipe_custom.fit(X_noisy[:160], y_data[:160])
score = pipe_custom.score(X_noisy[160:], y_data[160:])
print(f"Custom pipeline with outlier clipping: {score:.4f}")

# Also works with cross_val_score
cv_score = cross_val_score(pipe_custom, X_noisy, y_data, cv=5).mean()
print(f"5-fold CV: {cv_score:.4f}")

### Exercise 8.2 — Custom Transformer
Build a `FrequencyEncoder` transformer that:
- Encodes each categorical feature by its frequency in the training data
- e.g., if 'cat' appears 40% of the time, encode 'cat' as 0.40
- Unseen categories get frequency 0 (not NaN)
- Works with `BaseEstimator, TransformerMixin` so it fits in Pipelines

Test with a simple string array.

In [ ]:
# YOUR CODE HERE


In [ ]:
# SOLUTION
class FrequencyEncoder(BaseEstimator, TransformerMixin):
    """Encodes categorical features as their frequency in the training set."""
    
    def fit(self, X, y=None):
        X = np.array(X)
        self.freq_maps_ = []
        for col in range(X.shape[1] if X.ndim > 1 else 1):
            col_data = X[:, col] if X.ndim > 1 else X
            values, counts = np.unique(col_data, return_counts=True)
            freq_map = dict(zip(values, counts / len(col_data)))
            self.freq_maps_.append(freq_map)
        return self
    
    def transform(self, X, y=None):
        check_is_fitted(self, 'freq_maps_')
        X = np.array(X)
        result = np.zeros_like(X, dtype=float)
        for col in range(X.shape[1] if X.ndim > 1 else 1):
            col_data = X[:, col] if X.ndim > 1 else X
            result[:, col] if X.ndim > 1 else result
            encoded = np.array([self.freq_maps_[col].get(v, 0.0) for v in col_data])
            if X.ndim > 1:
                result[:, col] = encoded
            else:
                result = encoded
        return result

# Test
train_cats = np.array([['cat'], ['dog'], ['cat'], ['bird'], ['dog'], ['cat'],
                       ['bird'], ['cat'], ['dog'], ['fish']])
test_cats = np.array([['cat'], ['snake'], ['bird'], ['dog']])

enc = FrequencyEncoder()
enc.fit(train_cats)
print("Train frequency map:", enc.freq_maps_[0])
print("Train encoded:", enc.transform(train_cats).flatten())
print("Test encoded:", enc.transform(test_cats).flatten(), "← 'snake'=0 (unseen)")

---
## 4. Custom Sklearn-Compatible Estimator

In [ ]:
# Full custom classifier that integrates with sklearn ecosystem
class ThresholdClassifier(BaseEstimator, ClassifierMixin):
    """Classifies based on a single feature threshold.
    
    Finds the best feature + threshold to minimize Gini impurity.
    Like a depth-1 decision tree (decision stump).
    """
    
    def __init__(self):
        pass
    
    def fit(self, X, y):
        X, y = check_X_y(X, y)
        self.classes_ = np.unique(y)
        
        best_gini = float('inf')
        self.best_feature_ = 0
        self.best_threshold_ = 0
        self.best_direction_ = 1
        
        n = len(y)
        for feat in range(X.shape[1]):
            thresholds = np.unique(X[:, feat])
            for thresh in thresholds:
                for direction in [1, -1]:
                    pred = np.where(direction * X[:, feat] >= direction * thresh, 1, 0)
                    gini = self._gini(y[pred==0]) * (pred==0).sum()/n + \
                           self._gini(y[pred==1]) * (pred==1).sum()/n
                    if gini < best_gini:
                        best_gini = gini
                        self.best_feature_ = feat
                        self.best_threshold_ = thresh
                        self.best_direction_ = direction
        return self
    
    def _gini(self, y):
        if len(y) == 0:
            return 0
        _, counts = np.unique(y, return_counts=True)
        p = counts / len(y)
        return 1 - np.sum(p**2)
    
    def predict(self, X):
        check_is_fitted(self)
        X = check_array(X)
        return np.where(
            self.best_direction_ * X[:, self.best_feature_] >= 
            self.best_direction_ * self.best_threshold_, 1, 0
        )
    
    def __repr__(self):
        if hasattr(self, 'best_feature_'):
            return (f"ThresholdClassifier(feature={self.best_feature_}, "
                    f"threshold={self.best_threshold_:.4f})")
        return "ThresholdClassifier(not fitted)"

# Works with all sklearn tools!
stump = ThresholdClassifier()
stump.fit(X_tr, y_tr)
print(stump)
print(f"Train accuracy: {stump.score(X_tr, y_tr):.4f}")
print(f"Test accuracy:  {stump.score(X_te, y_te):.4f}")

# Cross-validation works!
cv_scores = cross_val_score(ThresholdClassifier(), X_data, y_data, cv=5)
print(f"5-fold CV: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

---
## 5. PyTorch-Style Dataset Class

In [ ]:
# Implement a Dataset class (same interface as PyTorch's Dataset)
class TabularDataset:
    """A PyTorch-style Dataset wrapper for tabular data."""
    
    def __init__(self, X, y, transform=None):
        self.X = np.array(X, dtype=float)
        self.y = np.array(y)
        self.transform = transform
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        x = self.X[idx]
        y = self.y[idx]
        if self.transform:
            x = self.transform(x)
        return x, y
    
    def __repr__(self):
        return f"TabularDataset(n_samples={len(self)}, n_features={self.X.shape[1]})"
    
    def get_batch(self, indices):
        """Get a batch by indices."""
        batch_X = np.array([self[i][0] for i in indices])
        batch_y = np.array([self[i][1] for i in indices])
        return batch_X, batch_y
    
    def class_weights(self):
        """Compute inverse class frequency weights for imbalanced data."""
        classes, counts = np.unique(self.y, return_counts=True)
        weights = len(self.y) / (len(classes) * counts)
        return dict(zip(classes, weights))


class DataLoader:
    """Iterates over a Dataset in batches (simplified PyTorch DataLoader)."""
    
    def __init__(self, dataset, batch_size=32, shuffle=True):
        self.dataset = dataset
        self.batch_size = batch_size
        self.shuffle = shuffle
    
    def __iter__(self):
        indices = np.arange(len(self.dataset))
        if self.shuffle:
            np.random.shuffle(indices)
        for start in range(0, len(indices), self.batch_size):
            batch_idx = indices[start:start + self.batch_size]
            yield self.dataset.get_batch(batch_idx)
    
    def __len__(self):
        return int(np.ceil(len(self.dataset) / self.batch_size))
    
    def __repr__(self):
        return f"DataLoader(n_batches={len(self)}, batch_size={self.batch_size})"


# Normalize transform
mean = X_data.mean(axis=0)
std  = X_data.std(axis=0)
normalize = lambda x: (x - mean) / (std + 1e-8)

dataset = TabularDataset(X_data, y_data, transform=normalize)
loader  = DataLoader(dataset, batch_size=32, shuffle=True)

print(dataset)
print(loader)
print(f"Class weights: {dataset.class_weights()}")

# Simulate 1 epoch
batch_sizes = []
for batch_X, batch_y in loader:
    batch_sizes.append(len(batch_X))

print(f"Batch sizes: {batch_sizes} (last batch may be smaller)")
print(f"Total samples processed: {sum(batch_sizes)}")

---
## Challenge: Build an Ensemble Classifier

Implement a **VotingEnsemble** class that:
1. Takes a list of `(name, estimator)` tuples
2. `fit(X, y)` — fits each estimator
3. `predict(X)` — majority vote (hard voting)
4. `predict_proba(X)` — average probabilities (soft voting)
5. `score(X, y)` — accuracy using hard voting
6. Must be compatible with `cross_val_score`
7. Compare it against each individual model and the sklearn `VotingClassifier`

In [ ]:
# YOUR CODE HERE


In [ ]:
# SOLUTION
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import VotingClassifier

class VotingEnsemble(BaseEstimator, ClassifierMixin):
    """Hard and soft voting ensemble of sklearn-compatible estimators."""
    
    def __init__(self, estimators, voting='hard'):
        self.estimators = estimators
        self.voting = voting
    
    def fit(self, X, y):
        X, y = check_X_y(X, y)
        self.classes_ = np.unique(y)
        self.estimators_ = []
        for name, est in self.estimators:
            fitted = clone(est).fit(X, y)
            self.estimators_.append((name, fitted))
        return self
    
    def predict_proba(self, X):
        check_is_fitted(self, 'estimators_')
        probas = np.array([est.predict_proba(X) for _, est in self.estimators_])
        return probas.mean(axis=0)
    
    def predict(self, X):
        check_is_fitted(self, 'estimators_')
        if self.voting == 'soft':
            return self.classes_[np.argmax(self.predict_proba(X), axis=1)]
        # Hard voting
        all_preds = np.array([est.predict(X) for _, est in self.estimators_])
        return np.apply_along_axis(
            lambda col: np.bincount(col, minlength=len(self.classes_)).argmax(),
            axis=0, arr=all_preds
        )
    
    def __repr__(self):
        names = [n for n, _ in self.estimators]
        return f"VotingEnsemble(estimators={names}, voting={self.voting!r})"


# Build estimators
estimators = [
    ('lr',  Pipeline([('s', StandardScaler()), ('m', LogisticRegression(max_iter=1000, random_state=42))])),
    ('rf',  RandomForestClassifier(n_estimators=50, random_state=42)),
    ('knn', Pipeline([('s', StandardScaler()), ('m', KNeighborsClassifier(n_neighbors=5))]))
]

my_ensemble    = VotingEnsemble(estimators, voting='soft')
sklearn_voting = VotingClassifier(estimators, voting='soft')

cv = 5
print(f"{'Model':<20} {'CV Accuracy':>15}")
print("-" * 38)

for name, model in [('LogReg', estimators[0][1]),
                     ('RandomForest', estimators[1][1]),
                     ('KNN', estimators[2][1]),
                     ('My VotingEnsemble', my_ensemble),
                     ('Sklearn Voting', sklearn_voting)]:
    scores = cross_val_score(model, X_data, y_data, cv=cv, scoring='accuracy')
    print(f"{name:<20} {scores.mean():.4f} ± {scores.std():.4f}")